# Email triage, cell by cell

**Synthetic data.** The companies are real; every email, sender, position, thesis and estimate is invented. Nothing here is investment advice.

This notebook runs the pipeline's real stage functions on three emails from `day_1`, one stage per cell:

1. **Thesis relevant:** Jev labels it `thesis_relevant`, the analysis agent suggests a change to an existing thesis, code validates it, and the analyst accepts it into the book.
2. **Human attention:** Jev flags it for a person, and a small model writes an attention note with a summary, the reason and the action asked for.
3. **Irrelevant:** Jev labels it `irrelevant`, and the gate stops it with a reason, so no further model sees it.

Models recommend; only the analyst changes the book. Every model call goes through the project's disk cache, so after a pipeline run on `day_1` this notebook replays at no cost.

## Setup

In [1]:
import json
import os

os.environ["TQDM_DISABLE"] = "1"                   # no model-loading progress bars in the outputs
from IPython.display import Markdown, display

from triage_app import config, thresholds
from triage_app.corpus import load
from triage_app.monitoring import format_summary, recording
from triage_app.pipeline import analyze, classify, extract, gate, human_attention, merge, redundancy, validate
from triage_app.pipeline.context import RunContext
from triage_app.state import apply
from triage_app.state.compute import compute
from triage_app.state.fold import fold, load_seed

CORPUS_SET = "day_1"
ctx = RunContext(CORPUS_SET, emails_path=config.corpus_file(CORPUS_SET))
recording(ctx.recorder).__enter__()          # record tokens and latency for every call below
EMAILS = {e.email_id: e for e in ctx.emails}  # label-free, in arrival order
SEED = load_seed()
print(f"{len(EMAILS)} emails in {CORPUS_SET}; Jev model {config.JEV_MODEL}; "
      f"analysis model {config.ANALYSIS_MODEL}; notes model {config.NOTES_MODEL}")

300 emails in day_1; Jev model jev-1.13.0; analysis model anthropic/claude-haiku-5.5; notes model anthropic/claude-haiku-5.5


In [2]:
def show(model):
    """Pretty-print a contract (a Pydantic model or a list of them)."""
    data = [m.model_dump(mode="json") for m in model] if isinstance(model, list) else model.model_dump(mode="json")
    print(json.dumps(data, indent=2, ensure_ascii=False))


def show_email(email, chars=1500):
    body = email.body if len(email.body) <= chars else email.body[:chars] + " …"
    display(Markdown(f"**From:** {email.sender} <{email.sender_email}>  \n**Subject:** {email.subject}  \n"
                     f"**Received:** {email.received_at:%Y-%m-%d %H:%M}"))
    print(body)


def show_triage(t):
    top = sorted(t.triage_probs.items(), key=lambda kv: -kv[1])
    print("triage:          ", ", ".join(f"{k} {v:.2f}" for k, v in top))
    print("signal score:    ", round(t.triage_probs["thesis_relevant"] + t.triage_probs["monitor"], 3))
    print("email_type:      ", t.email_type, f"({t.email_type_probs[t.email_type]:.2f})")
    print("human_attention: ", round(t.human_attention, 3))
    print("tickers:         ", {k: round(v, 2) for k, v in t.ticker_probs.items() if v >= 0.05})
    print("topics:          ", {k: round(v, 2) for k, v in t.topic_probs.items() if v >= 0.05})
    print("safety:          ", {k: round(v, 2) for k, v in t.safety.items()})
    print("criteria version:", t.criteria_version, "| truncated:", t.truncated)


def stage(name):
    """Time a cell as one stage for the current email, so the token table below is per stage."""
    return ctx.recorder.stage(name, email.email_id)


def earlier_than(email):
    """The day's emails that arrived before this one: the repeat check's day cache."""
    return [e for e in ctx.emails if e.received_at < email.received_at]


def corpus_label(email_id):
    """The corpus label, read here only to compare with the pipeline. No stage ever sends it to a model."""
    labels = {lb.email_id: lb for lb in load(config.corpus_file(CORPUS_SET))[1]}
    lb = labels[email_id]
    return {"triage": lb.triage, "human_attention": lb.human_attention, "affected_tickers": lb.affected_tickers,
            "email_type": lb.email_type}

---
## Case 1: a thesis-relevant email changes a thesis

The email passes the gate, so it takes the analysis pathway: extract claims, let the analysis agent call a skill, validate the suggestion in code, merge, and put it in front of the analyst.

In [3]:
THESIS_EMAIL = "synthetic_000261"

### Case 1: the email

Every stage reads the email from the corpus file through the loader, which drops every label. This is exactly what the models see: sender, address, subject and body.

In [4]:
email = EMAILS[THESIS_EMAIL]
show_email(email)

**From:** Tomas Reyes, Account Director, Eastgate Display Solutions <treyes@eastgatedisplay.com>  
**Subject:** quick note - Pro Max OLED panel orders  
**Received:** 2026-10-13 17:12

Raj,

Hope you're well. Following up on our chat at the Taipei display show. Figured you'd want this before the Apple print since it fits what you were asking about.

Our team supplies driver-IC and touch assemblies into a couple of the panel makers that feed the iPhone 18 Pro / Pro Max. Over the past three weeks, purchase orders for Pro Max-class panels have been pulled forward. Two panel customers raised their December-quarter orders with us by roughly 9-12% versus the plan we had in early September, and one asked for expedited shipment of the October lot. Base model orders were flat.

What I'd note:
- The increase is on the Pro Max in particular, and the larger-size tooling lines are running close to full.
- One customer told us the upward revision came from Apple's side after strong pre-order conversion in the U.S. and Japan, with fewer cancellations than last year.
- Our own lead times on the touch ICs have stretched from 6 to 8 weeks, which tells me demand is firm and not just in

### Stage 2: check for repeats

The email is embedded with the Hugging Face model and compared with every **earlier** email of the same day (cosine similarity), plus a token-set comparison of subjects. A flag is only a hint; stage 4 decides what to do with it.

In [5]:
with stage("redundancy"):
    day = redundancy.build_day_cache(earlier_than(email), ctx)
    repeat = redundancy.process(email, day, ctx)
show(repeat)
flagged = repeat if repeat.flagged else None

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (538 > 512). Running this sequence through the model will result in indexing errors


{
  "email_id": "synthetic_000261",
  "nearest": "synthetic_000001",
  "content_similarity": 0.8645973205566406,
  "subject_score": 0.3211678832116789,
  "flagged": true
}


### Stage 3: classify with Jev

One request with all 14 questions. Jev receives the email and the desk's relevance criteria (`criteria/*.md`) and nothing else: no book, no positions, no other emails, no repeat flag. It returns probabilities, not decisions.

In [6]:
with stage("classify"):
    triage = classify.process(email, ctx)
show_triage(triage)

triage:           thesis_relevant 0.81, monitor 0.17, low_value 0.01, redundant 0.00, irrelevant 0.00
signal score:     0.98
email_type:       primary_research (1.00)
human_attention:  0.39
tickers:          {'AMZN': 0.05, 'AAPL': 0.91, 'GOOGL': 0.05}
topics:           {'sector': 0.6, 'other': 0.86}
safety:           {'possible_mnpi': 0.59, 'instructs_ai': 0.03}
criteria version: 1f44a2faac8e | truncated: False


### Stage 4: label and gate (code, no model)

Thresholds from `thresholds.py` (or `tuned/thresholds.json`) turn the probabilities into the label of record, the companies and topics, the human-attention flag and the gate. The one-line reason is composed by code.

In [7]:
with stage("gate"):
    result = gate.process(triage, flagged, ctx.thresholds)
show(result)

{
  "email_id": "synthetic_000261",
  "triage": "thesis_relevant",
  "decided_by": "jev",
  "redundant_of": null,
  "affected_tickers": [
    "AAPL"
  ],
  "additional_labels": [
    "sector",
    "other"
  ],
  "human_attention": false,
  "signal_score": 0.98,
  "gate": "pass",
  "reason": "thesis_relevant 0.81; signal score 0.98 clears 0.98"
}


### Stage 5: extract claims

A schema-constrained call to the analysis model. Each claim carries a verbatim quote; code checks every quote against the email by string match and drops any that fail. Code assigns the claim IDs.

In [8]:
earlier = EMAILS.get(repeat.nearest) if repeat.flagged and repeat.nearest else None
with stage("extract"):
    claims = extract.process(email, result, earlier, ctx)
show(claims)

[
  {
    "quote": "Two panel customers raised their December-quarter orders with us by roughly 9-12% versus the plan we had in early September",
    "tickers": [
      "AAPL"
    ],
    "entities": [],
    "kind": "channel_check",
    "metric": "Pro Max-class panel orders",
    "period": "December quarter",
    "value": null,
    "unit": "pct",
    "direction": "up",
    "first_hand": true,
    "id": "synthetic_000261.c1",
    "email_id": "synthetic_000261"
  },
  {
    "quote": "The increase is on the Pro Max in particular, and the larger-size tooling lines are running close to full.",
    "tickers": [
      "AAPL"
    ],
    "entities": [],
    "kind": "channel_check",
    "metric": "Pro Max panel orders",
    "period": null,
    "value": null,
    "unit": null,
    "direction": "up",
    "first_hand": true,
    "id": "synthetic_000261.c2",
    "email_id": "synthetic_000261"
  },
  {
    "quote": "Base model orders were flat.",
    "tickers": [
      "AAPL"
    ],
    "entities": []

### Stage 6: the analysis agent

Code picks the candidate pillars (the claims' companies plus the read-through links). The agent sees the claims and those pillars' statements, and chooses which skill to call (at most three calls): **alter an existing thesis** or **spawn a new one**. Suggestions come from the skills' structured output, never from the agent's free text. Code fills the book's current value and consensus beside any figure the email states.

In [9]:
book = fold(SEED, [])
with stage("analyze"):
    record, raw_suggestions = analyze.process(email.email_id, claims, result, ctx, book=book)
print("skills called:", record.skills_called, "| no-change reason:", record.no_change_reason)
show(raw_suggestions)

skills called: ['existing_thesis'] | no-change reason: None
[
  {
    "id": "synthetic_000261.s1",
    "body": {
      "kind": "existing_thesis",
      "pillar_id": "AAPL.p1",
      "stance": "contradicts",
      "strength": 2,
      "wrong_if_met": false,
      "assumptions": []
    },
    "rationale": "Channel checks report raised December-quarter iPhone panel orders and near-full tooling lines, which cut against the pillar's claim that the upgrade cycle disappoints, though they are not sell-through data and do not compare with consensus.",
    "claim_ids": [
      "synthetic_000261.c1",
      "synthetic_000261.c2",
      "synthetic_000261.c4"
    ],
    "sections": [
      {
        "email_id": "synthetic_000261",
        "quote": "Two panel customers raised their December-quarter orders with us by roughly 9-12% versus the plan we had in early September"
      },
      {
        "email_id": "synthetic_000261",
        "quote": "The increase is on the Pro Max in particular, and the l

### Stages 7 and 8: validate and merge (code, no model)

Validation rejects unknown IDs, quotes that are not in the email, and duplicate theses. It drops a stated figure that is out of bounds, for the wrong period, or not written in the claim's quote, and caps monitor-only evidence at strength 1. Merging then gives one suggestion per pillar and stance.

In [10]:
inputs = validate.ValidationInputs(claims, list(ctx.emails), [result], SEED)
with stage("validate"):
    checked = [validate.process(s, ctx, inputs) for s in raw_suggestions]
for s in checked:
    print(s.id, "->", s.status, s.reject_reason or "")
with stage("merge"):
    suggestions = merge.process(checked, ctx, [result], SEED)
show(suggestions)

synthetic_000261.s1 -> open 
[
  {
    "id": "AAPL.p1.contradicts",
    "body": {
      "kind": "existing_thesis",
      "pillar_id": "AAPL.p1",
      "stance": "contradicts",
      "strength": 2,
      "wrong_if_met": false,
      "assumptions": []
    },
    "rationale": "Channel checks report raised December-quarter iPhone panel orders and near-full tooling lines, which cut against the pillar's claim that the upgrade cycle disappoints, though they are not sell-through data and do not compare with consensus.",
    "claim_ids": [
      "synthetic_000261.c1",
      "synthetic_000261.c2",
      "synthetic_000261.c4"
    ],
    "sections": [
      {
        "email_id": "synthetic_000261",
        "quote": "Two panel customers raised their December-quarter orders with us by roughly 9-12% versus the plan we had in early September"
      },
      {
        "email_id": "synthetic_000261",
        "quote": "The increase is on the Pro Max in particular, and the larger-size tooling lines are ru

### The analyst decides

Only an accepted suggestion changes the book, through the change log. Accepting logs the evidence against the pillar with its quoted sections. When the suggestion shows a figure the email states for a linked driver, the analyst can also update that assumption, and code recomputes EPS and the target price.

In [11]:
suggestion = suggestions[0]
log = [apply.accept(SEED, [], suggestion)]
state = fold(SEED, log)
body = suggestion.body
pillar_id = getattr(body, "pillar_id", None)
print("logged:", log[0].change, log[0].item_id, getattr(log[0], "stance", None), "strength", log[0].strength)
if pillar_id:
    pillar = next(p for t in state.theses.values() for p in t.pillars if p.id == pillar_id)
    print("\npillar:", pillar.statement)
    print("wrong if:", pillar.wrong_if)
    print("accepted evidence on this pillar:", len(state.evidence[pillar_id]))

logged: pillar_evidence AAPL.p1 contradicts strength 2

pillar: The iPhone upgrade cycle disappoints: AI features do not pull upgrades forward, and share in China stays under pressure.
wrong if: Sell-through or lead-time data show a clearly stronger cycle than consensus expects.
accepted evidence on this pillar: 1


In [12]:
ticker = (pillar_id or getattr(body, "ticker", "")).split(".")[0]
before = compute(fold(SEED, []).models[ticker])["analyst"]
assumptions = getattr(body, "assumptions", [])
if assumptions:
    a = assumptions[0]
    print(f"email states {a.driver_id} = {a.stated_value} (book {a.book_value}, consensus {a.consensus_value})")
    log.append(apply.update_driver(SEED, log, a.driver_id, a.stated_value, suggestion_id=suggestion.id))
    after = compute(fold(SEED, log).models[ticker])["analyst"]
    print(f"{ticker} EPS {before.eps:.2f} -> {after.eps:.2f}; "
          f"target price {before.target_price:.0f} -> {after.target_price:.0f}")
else:
    print(f"The email states no figure for a driver linked to this pillar, so no assumption changes: "
          f"{ticker} EPS stays {before.eps:.2f}, target price {before.target_price:.0f}. "
          f"The book changed only by the logged evidence, which counts toward a conviction review.")

The email states no figure for a driver linked to this pillar, so no assumption changes: AAPL EPS stays 7.27, target price 196. The book changed only by the logged evidence, which counts toward a conviction review.


In [13]:
corpus_label(THESIS_EMAIL)

{'triage': 'thesis_relevant',
 'human_attention': False,
 'affected_tickers': ['AAPL'],
 'email_type': 'primary_research'}

---
## Case 2: an email that needs a person

Jev's human-attention probability clears its threshold, so the email takes the attention pathway. A small generative model writes a note: a two-sentence summary, why it needs a person, the action asked for and any deadline, with supporting quotes that code checks against the email.

In [14]:
ATTENTION_EMAIL = "synthetic_000144"

### Case 2: the email

Every stage reads the email from the corpus file through the loader, which drops every label. This is exactly what the models see: sender, address, subject and body.

In [15]:
email = EMAILS[ATTENTION_EMAIL]
show_email(email)

**From:** Natalia Brandvold, Head of Expert Programs, Tessellate Advisory Network <nbrandvold@tessellateadvisory.com>  
**Subject:** Small-group call Wed 7:45am ET: ex-AI lab capacity planner on 2027 inference sourcing (GPU vs. TPU) - early read, 4 seats  
**Received:** 2026-10-13 11:56

Hi Ellen,

I'm flagging a small-group call we have arranged for tomorrow morning. Raj and Marcus are welcome to join too, but we are capping this at four seats in total.

Guest: Former Head of Compute Capacity Planning at a frontier AI lab (left Q2 2026, now independent consultant advising two data center developers). He was involved in capacity sourcing for several years, including multi-vendor negotiations.

Why we are calling now: In our pre-screen, he said two former colleagues, both still at AI labs, told him in recent weeks that their organizations are asking for committed TPU capacity from Google Cloud for 2027 inference. The commitments would be a meaningful share of planned inference growth that had previously been assumed to be filled with Nvidia systems. He was explicit that this is secondhand. He has not seen contracts or volumes, and he does not know whether it is a hedging exercise, a negotiating tactic or a real shift. He also said that training demand appears unaffected

### Stage 2: check for repeats

The email is embedded with the Hugging Face model and compared with every **earlier** email of the same day (cosine similarity), plus a token-set comparison of subjects. A flag is only a hint; stage 4 decides what to do with it.

In [16]:
with stage("redundancy"):
    day = redundancy.build_day_cache(earlier_than(email), ctx)
    repeat = redundancy.process(email, day, ctx)
show(repeat)
flagged = repeat if repeat.flagged else None

{
  "email_id": "synthetic_000144",
  "nearest": "synthetic_000136",
  "content_similarity": 0.8511579036712646,
  "subject_score": 0.4403669724770642,
  "flagged": true
}


### Stage 3: classify with Jev

One request with all 14 questions. Jev receives the email and the desk's relevance criteria (`criteria/*.md`) and nothing else: no book, no positions, no other emails, no repeat flag. It returns probabilities, not decisions.

In [17]:
with stage("classify"):
    triage = classify.process(email, ctx)
show_triage(triage)

triage:           monitor 0.98, thesis_relevant 0.01, low_value 0.01, redundant 0.00, irrelevant 0.00
signal score:     0.99
email_type:       meeting_request (0.96)
human_attention:  0.96
tickers:          {'AMZN': 0.13, 'NVDA': 0.86, 'MSFT': 0.09, 'AAPL': 0.06, 'GOOGL': 0.83}
topics:           {'sector': 0.92, 'other': 0.53}
safety:           {'possible_mnpi': 0.09, 'instructs_ai': 0.04}
criteria version: 1f44a2faac8e | truncated: False


### Stage 4: label and gate (code, no model)

Thresholds from `thresholds.py` (or `tuned/thresholds.json`) turn the probabilities into the label of record, the companies and topics, the human-attention flag and the gate. The one-line reason is composed by code.

In [18]:
with stage("gate"):
    result = gate.process(triage, flagged, ctx.thresholds)
show(result)

{
  "email_id": "synthetic_000144",
  "triage": "monitor",
  "decided_by": "jev",
  "redundant_of": null,
  "affected_tickers": [
    "NVDA",
    "GOOGL"
  ],
  "additional_labels": [
    "sector",
    "other"
  ],
  "human_attention": true,
  "signal_score": 0.99,
  "gate": "pass",
  "reason": "monitor 0.98; signal score 0.99 clears 0.98"
}


### Stage A: the attention note

The note explains the flag; it cannot remove it, and it never recommends an investment action.

In [19]:
with stage("human_attention"):
    note = human_attention.process(email, result, ctx)
display(Markdown(f"**Summary.** {note.summary}\n\n**Why it needs a person.** {note.why_attention}\n\n"
                 f"**Action:** {note.action}  \n**Deadline:** {note.deadline or 'none stated'}"))
for s in note.sections:
    print("quote:", s.quote)

**Summary.** Tessellate Advisory Network is offering a small-group call on Wednesday 7:45-8:30am ET with a former Head of Compute Capacity Planning at a frontier AI lab. The topic is whether 2027 inference sourcing is shifting toward Google Cloud TPUs, and the sender describes the signal as early and unconfirmed.

**Why it needs a person.** Seats are capped at four in total and the sender needs a confirmation today to hold one, so a reply is needed before the deadline. The call's subject is relevant to the Nvidia inference-mix and Google Cloud capacity reports, though the sender says the underlying information is secondhand.

**Action:** reply  
**Deadline:** 2026-10-13 17:00:00

quote: we are capping this at four seats in total.
quote: Please confirm by 5pm today so I can hold your seats.


In [20]:
corpus_label(ATTENTION_EMAIL)

{'triage': 'monitor',
 'human_attention': True,
 'affected_tickers': ['NVDA', 'GOOGL'],
 'email_type': 'event_invitation'}

---
## Case 3: an irrelevant email is stopped

Jev labels it `irrelevant` and its signal score is below the pass threshold, so the gate stops it. No further model reads it; it appears only in the audit view, with the reason below.

In [21]:
IRRELEVANT_EMAIL = "synthetic_000007"

### Case 3: the email

Every stage reads the email from the corpus file through the loader, which drops every label. This is exactly what the models see: sender, address, subject and body.

In [22]:
email = EMAILS[IRRELEVANT_EMAIL]
show_email(email)

**From:** Dr. Hannah Voss, Senior Analyst, Medical Devices, Corrigan Whitlock Securities <hvoss@corriganwhitlock.com>  
**Subject:** MedTech: Bariatric procedure volumes soften as GLP-1 scripts climb - 3Q preview and estimate changes  
**Received:** 2026-10-13 05:46

Corrigan Whitlock Securities | Healthcare Equity Research | Industry Update

Bariatric surgery: volumes decelerating faster than we modeled

We update our procedure-volume model ahead of 3Q reporting season (first prints from our coverage start Oct 20).

What our data show:
- Our hospital claims panel (about 640 U.S. facilities) shows sleeve gastrectomy and gastric bypass volumes down 9.4% y/y in August, versus -6.1% in 1H26.
- Weekly GLP-1 prescriptions for obesity are up roughly 38% y/y, with the sharpest growth in the 35-45 BMI cohort that historically fed the surgical funnel.
- Consult-to-surgery conversion at the 40 largest centers fell to 41% from 47% a year ago. Surgeons say patients increasingly try drug therapy for 6-9 months before booking.
- Offsetting: revision procedures (patients regaining weight after drug discontinuation) are +14% y/y, though from a small base.

Estimate changes:
- Stapler and energy-device vendors: we trim 3Q bariatric revenue by 3-5% and 4Q by 5-7%. B

### Stage 2: check for repeats

The email is embedded with the Hugging Face model and compared with every **earlier** email of the same day (cosine similarity), plus a token-set comparison of subjects. A flag is only a hint; stage 4 decides what to do with it.

In [23]:
with stage("redundancy"):
    day = redundancy.build_day_cache(earlier_than(email), ctx)
    repeat = redundancy.process(email, day, ctx)
show(repeat)
flagged = repeat if repeat.flagged else None

{
  "email_id": "synthetic_000007",
  "nearest": "synthetic_000005",
  "content_similarity": 0.7426431179046631,
  "subject_score": 0.4210526315789474,
  "flagged": false
}


### Stage 3: classify with Jev

One request with all 14 questions. Jev receives the email and the desk's relevance criteria (`criteria/*.md`) and nothing else: no book, no positions, no other emails, no repeat flag. It returns probabilities, not decisions.

In [24]:
with stage("classify"):
    triage = classify.process(email, ctx)
show_triage(triage)

triage:           irrelevant 0.91, thesis_relevant 0.04, monitor 0.04, low_value 0.01, redundant 0.00
signal score:     0.08
email_type:       sell_side_research (1.00)
human_attention:  0.09
tickers:          {'AMZN': 0.05, 'GOOGL': 0.06}
topics:           {'sector': 0.98, 'government': 0.06, 'other': 0.45}
safety:           {'possible_mnpi': 0.03, 'instructs_ai': 0.02}
criteria version: 1f44a2faac8e | truncated: False


### Stage 4: label and gate (code, no model)

Thresholds from `thresholds.py` (or `tuned/thresholds.json`) turn the probabilities into the label of record, the companies and topics, the human-attention flag and the gate. The one-line reason is composed by code.

In [25]:
with stage("gate"):
    result = gate.process(triage, flagged, ctx.thresholds)
show(result)

{
  "email_id": "synthetic_000007",
  "triage": "irrelevant",
  "decided_by": "jev",
  "redundant_of": null,
  "affected_tickers": [],
  "additional_labels": [
    "sector",
    "other"
  ],
  "human_attention": false,
  "signal_score": 0.08,
  "gate": "stop",
  "reason": "irrelevant 0.91; signal score 0.08 is below 0.98"
}


### After the gate

Stopped emails make no further model calls: extraction returns nothing, and no note is written unless Jev flagged the email for a person.

In [26]:
print("gate:", result.gate, "| human attention:", result.human_attention)
print("claims extracted:", extract.process(email, result, None, ctx))

gate: stop | human attention: False
claims extracted: []


In [27]:
corpus_label(IRRELEVANT_EMAIL)

{'triage': 'irrelevant',
 'human_attention': False,
 'affected_tickers': [],
 'email_type': 'sell_side_research'}

---
## Token use and latency for this notebook

Every call above was recorded. Cache hits cost nothing; the spent and uncached columns show the difference.

In [28]:
print(format_summary(ctx.recorder.report(CORPUS_SET, emails=3)))

stage             calls  hits    in tok  out tok   p50 ms   p95 ms
analyze               3     3         0        0     61.2     61.2
classify              3     3         0        0      0.7     15.4
extract               1     1         0        0     60.2     60.2
gate                  0     0         0        0      0.0      0.0
human_attention       1     1         0        0     25.1     25.1
merge                 0     0         0        0      0.0      0.0
redundancy          425   425         0        0    413.6   5908.6
validate              0     0         0        0      0.2      0.2
3 emails; 0 tokens/email spent; email latency p50 439 ms, p95 6036 ms; total 7.1 s
